# Modèle lourd : DistilBERT mesuré par CodeCarbon (Kaggle) · Ons

Chaîne complète : données (code commun) → DistilBERT, 1 époque → accuracy + `emissions.csv`.

**Réglages Kaggle avant d'exécuter** (panneau de droite, *Settings*) :
1. *Accelerator* : **GPU T4 x2** (ou P100). On n'en utilise qu'un seul (cellule 2) pour une mesure propre.
2. *Internet* : **On** (téléphone vérifié requis), sinon pas de téléchargement d'AG News ni de DistilBERT.
3. *Add-ons → Secrets* : ajouter `GITHUB_TOKEN` (jeton GitHub avec accès au dépôt privé) et l'attacher au notebook.

In [ ]:
!nvidia-smi

Un seul GPU : sinon le Trainer répartit sur les deux T4 et la mesure n'est plus comparable. À faire **avant** d'importer torch.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [ ]:
from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret("GITHUB_TOKEN")
!git clone -q https://{token}@github.com/onsalouini/green-ai-carbone.git
%cd /kaggle/working/green-ai-carbone
!git log --oneline -1

Kaggle fournit déjà torch, transformers, datasets et scikit-learn : on n'ajoute que CodeCarbon.

In [ ]:
!pip install -q codecarbon

## 1. Test court (500 exemples) : valider la mesure et estimer la durée du vrai run

In [ ]:
!python -m src.modele_lourd --test-court

In [ ]:
import json, pandas as pd
print(json.load(open("resultats/lourd_test_court.json"))["estimation_vrai_run_min"], "min estimées pour le vrai run")
pd.read_csv("emissions.csv")[["project_name", "duration", "energy_consumed", "emissions", "country_name", "region", "gpu_model"]]

## 2. Vrai run (10 000 / 2 000), seulement si l'estimation est < 20 min

In [ ]:
!python -m src.modele_lourd

In [ ]:
print(json.dumps(json.load(open("resultats/lourd.json")), indent=2, ensure_ascii=False))
pd.read_csv("emissions.csv")

## 3. Récupérer les résultats

Les fichiers sont dans `/kaggle/working/green-ai-carbone/` (`emissions.csv`, `resultats/*.json`) :
les télécharger depuis l'onglet *Output*, puis les committer en local dans `resultats/kaggle/`.